# Network Service Discovery

---

**MITRE ATT&CK** `T1046` · **Tactic:** Reconnaissance · **Difficulty:** Beginner · **Time:** ~40 min

> Before an attacker touches a single service, they listen. Port scanning is the act of knocking on every door in a building to see which ones open. It's the first thing that happens in almost every real-world intrusion — and it generates the most recognizable network noise.

---

### What you'll understand after this notebook

1. How the TCP three-way handshake is exploited in a SYN scan to enumerate open ports *without completing connections*
2. How Scapy constructs raw IP/TCP packets from scratch, bypassing the OS socket API
3. How to detect SYN scan activity in network logs and what distinguishes it from normal traffic
4. Why DNS port scanning is separate from TCP scanning — and what it reveals about a target's infrastructure

---

### Real-world context

In the 2020 SolarWinds breach, the Sunburst malware performed internal network reconnaissance after initial access — enumerating services before deciding which systems to pivot to. In the 2021 Colonial Pipeline attack, initial scanning of the VPN infrastructure identified exposed credentials. Port scanning isn't the sexy part of a breach story, but it's the part where attackers decide *where they're going next*.

## 1. The TCP Handshake — What a Normal Connection Looks Like

A standard TCP connection requires three steps:

```
Client  ──── SYN ────────────►  Server   "I want to connect"
Client  ◄─── SYN-ACK ─────────  Server   "OK, I'm here and listening"
Client  ──── ACK ────────────►  Server   "Great, connected."
```

If a port is **closed**:
```
Client  ──── SYN ────────────►  Server
Client  ◄─── RST-ACK ─────────  Server   "Nothing here, go away"
```

If a port is **filtered** (firewall drops the packet):
```
Client  ──── SYN ────────────►  [dropped]
Client  ◄── [nothing, timeout]
```

A **SYN scan** exploits this by only sending the first SYN packet and reading the response — never completing the handshake. The advantage: it's faster, leaves no application-layer log entry, and never establishes a full TCP session.

## 2. Environment Setup

Scapy needs raw socket access — run as root/sudo, or use a lab VM. Never scan hosts you don't own.

In [ ]:
from scapy.all import *       # raw packet construction and sending
import matplotlib.pyplot as plt  # visualization
import matplotlib.patches as mpatches
from collections import defaultdict

# Suppress Scapy's verbose output for cleaner notebook output
conf.verb = 0

print('Scapy version:', scapy.__version__)

## 3. Choosing What to Scan

The original script targets 7 ports. Let's understand *why* each one matters:

| Port | Service | Why attackers care |
|------|---------|-------------------|
| 25   | SMTP    | Email relay, phishing infrastructure, credential spray via SMTP AUTH |
| 53   | DNS     | Potential for zone transfers, DNS tunneling C2 |
| 80   | HTTP    | Exposed web services, unencrypted traffic to sniff |
| 443  | HTTPS   | Web apps — login portals, APIs |
| 445  | SMB     | Critical — EternalBlue lives here. File shares, lateral movement |
| 8080 | HTTP-alt| Developer/staging servers, often misconfigured |
| 8443 | HTTPS-alt| Same as above, often skips prod security controls |

In practice, attackers also scan 22 (SSH), 3389 (RDP), 1433 (MSSQL), 3306 (MySQL), and 5432 (PostgreSQL). For a full recon scan you'd use all of these.

In [ ]:
# Ports from the original script — common high-value services
ports = [25, 80, 53, 443, 445, 8080, 8443]

# Port-to-service lookup for readable output
PORT_NAMES = {
    25:   'SMTP',
    53:   'DNS',
    80:   'HTTP',
    443:  'HTTPS',
    445:  'SMB',
    8080: 'HTTP-alt',
    8443: 'HTTPS-alt',
    22:   'SSH',
    3389: 'RDP',
    1433: 'MSSQL',
}

## 4. The SYN Scan — Line by Line

Let's dissect the packet construction before running anything:

In [ ]:
# Let's build a single packet and inspect its structure before sending
# This is for learning — we don't send this cell

target = '8.8.8.8'  # Google DNS — safe, public, responds to pings

# Layer 1: IP header — source and destination
ip_layer = IP(dst=target)

# Layer 2: TCP header — destination port, SYN flag set
# flags="S" means only the SYN bit is set (hex 0x02)
tcp_layer = TCP(dport=80, flags='S')

# Stack them: IP / TCP (Scapy uses / as a layer separator)
packet = ip_layer / tcp_layer

# Show the packet structure
packet.show()

The `sr()` function (send-receive) sends a list of packets and returns two lists:
- `ans` — answered packets: tuples of `(sent_packet, received_response)`
- `unans` — unanswered packets: destinations that didn't respond (likely filtered)

When `dport` is a list, Scapy creates one packet per port automatically.

In [ ]:
def SynScan(host):
    """
    Performs a TCP SYN scan on a predefined list of ports.
    Returns a dict: {port: status} where status is 'open', 'closed', or 'filtered'
    """
    results = {}
    
    # Send SYN packets to all ports simultaneously, wait 2 seconds for responses
    # IP(dst=host) / TCP(dport=ports, flags='S') creates one packet per port
    ans, unans = sr(
        IP(dst=host) / TCP(dport=ports, flags='S'),
        timeout=2,
        verbose=0
    )
    
    # Process answered packets
    for sent, received in ans:
        port = sent[TCP].dport
        
        if received.haslayer(TCP):
            tcp_flags = received[TCP].flags
            
            # SYN-ACK (0x12) = port is OPEN and service is listening
            if tcp_flags == 0x12:  # SYN-ACK
                results[port] = 'open'
                # Send RST to cleanly close the half-open connection
                # (avoids leaving hanging connections on target)
                send(IP(dst=host) / TCP(dport=port, flags='R'), verbose=0)
            
            # RST-ACK (0x14) = port is CLOSED — service not running
            elif tcp_flags == 0x14:  # RST-ACK
                results[port] = 'closed'
    
    # Unanswered = filtered by firewall, no response at all
    for sent in unans:
        results[sent[TCP].dport] = 'filtered'
    
    return results


def DNSScan(host):
    """
    Checks whether a host is running a DNS resolver by sending a real DNS query.
    A valid DNS response indicates a resolver is present.
    """
    # Send a recursive DNS query (rd=1) for google.com over UDP port 53
    # Using 'google.com' as a benign test domain that should always resolve
    ans, _ = sr(
        IP(dst=host) / UDP(dport=53) / DNS(rd=1, qd=DNSQR(qname='google.com')),
        timeout=2,
        verbose=0
    )
    
    return bool(ans)  # True if we got any DNS response

## 5. Running the Scan

> **Lab note:** Use `127.0.0.1` or your local network range in a controlled environment. The scan below targets `8.8.8.8` (Google's public DNS resolver) for demonstration — it will show port 53 open and others filtered, which is the expected and safe result for a public resolver.

In [ ]:
host = '8.8.8.8'

print(f'Scanning {host}...\n')
scan_results = SynScan(host)
is_dns = DNSScan(host)

# Print results in a readable format
print(f'{'PORT':<8} {'SERVICE':<12} {'STATUS'}')
print('─' * 30)
for port in sorted(scan_results):
    service = PORT_NAMES.get(port, 'unknown')
    status = scan_results[port]
    marker = '●' if status == 'open' else '○'
    print(f'{marker} {port:<7} {service:<12} {status}')

print(f'\nDNS resolver present: {is_dns}')

## 6. Visualizing Scan Results

Raw output is fine for terminals. For a notebook or article, a chart communicates the same information faster.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0F0F0F')
ax.set_facecolor('#1A1A1A')

status_colors = {
    'open':     '#6C9BCF',  # blue
    'closed':   '#4A4540',  # muted
    'filtered': '#CF9B6C',  # amber
}

sorted_ports = sorted(scan_results.keys())
bar_colors = [status_colors[scan_results[p]] for p in sorted_ports]
bar_labels = [f"{p}\n{PORT_NAMES.get(p,'?')}" for p in sorted_ports]
bar_values = [1] * len(sorted_ports)  # uniform height — we care about color

bars = ax.bar(bar_labels, bar_values, color=bar_colors, width=0.6, edgecolor='#282828')

# Status labels inside bars
for bar, port in zip(bars, sorted_ports):
    status = scan_results[port]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        0.5,
        status,
        ha='center', va='center',
        color='#EDEAE4', fontsize=9,
        fontfamily='monospace'
    )

ax.set_yticks([])
ax.set_title(f'Port Scan Results — {host}', color='#EDEAE4', fontsize=13, pad=12)
ax.tick_params(colors='#7A7570')
for spine in ax.spines.values():
    spine.set_edgecolor('#282828')

# Legend
legend_elements = [mpatches.Patch(color=c, label=s) for s, c in status_colors.items()]
ax.legend(handles=legend_elements, loc='upper right',
          facecolor='#1A1A1A', edgecolor='#282828',
          labelcolor='#EDEAE4', fontsize=9)

plt.tight_layout()
plt.savefig('../cyberlab/content/articles/01_scan_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Defender's Perspective

### What this looks like in your logs

A SYN scan generates a distinctive signature:

```
# In a packet capture / IDS alert:
SRC_IP:random_port → DST_IP:25   [SYN]
SRC_IP:random_port → DST_IP:80   [SYN]     ← multiple ports,
SRC_IP:random_port → DST_IP:443  [SYN]        same source,
SRC_IP:random_port → DST_IP:445  [SYN]        rapid succession
SRC_IP:random_port → DST_IP:8080 [SYN]
```

### Indicators of Compromise (IOCs)

| Indicator | Threshold | Confidence |
|-----------|-----------|------------|
| Single IP → multiple destination ports in < 2 seconds | > 5 ports | High |
| SYN packets with no corresponding ACK or data | > 10/min from same src | High |
| RST packets sent immediately after SYN-ACK received | Any | Medium |
| Probes to known sensitive ports (445, 3389) | Any external src | High |

### Detection Tools
- **Suricata / Snort:** Rule `alert tcp any any -> $HOME_NET any (flags:S; threshold:type both, track by_src, count 10, seconds 2; msg:'SYN Scan';)`
- **Zeek:** Connection log shows many short-lived `S0` state connections (SYN sent, no response)
- **SIEM query (Splunk):** `index=network sourcetype=firewall action=blocked flags=SYN | stats count by src_ip | where count > 20`

## 8. Article Seed

---

**Suggested title:** *How Attackers Map Your Network Before They Attack — A Python Breakdown*

**Opening paragraph (edit this):**

> Before an attacker exploits a single vulnerability, they spend time listening. Port scanning — specifically the SYN scan technique — is the reconnaissance step that appears in virtually every documented breach. In this article, I'll walk through exactly how it works at the packet level, using Python and Scapy to build it from scratch, and then show what it looks like from the defender's side.

**Section headers:**
1. The TCP three-way handshake and why SYN scans exploit it
2. Building a port scanner with Scapy (raw packet construction)
3. What SYN scan traffic looks like in your IDS/SIEM

**Medium tags:** `cybersecurity, python, network-security, ethical-hacking, scapy`

---

## 9. Self-Check

In [ ]:
# Fill in the blanks — run this cell to check your understanding

# Q1: What TCP flag combination means a port is OPEN?
open_port_flags = None  # replace with the hex value
assert open_port_flags == 0x12, 'Hint: SYN + ACK = 0x02 + 0x10'

# Q2: What does Scapy return for unanswered packets?
unanswered_means = None  # replace with 'open', 'closed', or 'filtered'
assert unanswered_means == 'filtered', 'Unanswered = firewall dropped the packet'

# Q3: What does the / operator do in Scapy?
slash_operator = None  # replace with a short string description
assert 'layer' in slash_operator.lower(), 'Think: IP(dst=host) / TCP(dport=80)'

print('All checks passed. You understand SYN scanning.')